# ETL Bronze - ECMWF FC (Open Data)

Carga los JSON diarios de fc (todo el bounding box, sin recortar al poligono) en una tabla Bronze idempotente.

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, StringType, StructField, StructType
from pyspark.sql.window import Window

BRONZE_TABLE = 'weather.bronze.ecmwf_forecast_fc'
RAW_PATH = '/Volumes/weather/raw/ecmwf_volume/fc_opendata/json/'

schema = StructType([
    StructField('run_date', StringType(), True),
    StructField('run_time', StringType(), True),
    StructField('step_hours', IntegerType(), True),
    StructField('valid_datetime', StringType(), True),
    StructField('valid_date', StringType(), True),
    StructField('latitude', DoubleType(), True),
    StructField('longitude', DoubleType(), True),
    StructField('tp_mm', DoubleType(), True),
    StructField('tipo', StringType(), True),
    StructField('source_api', StringType(), True),
    StructField('extracted_at', StringType(), True),
])


In [ ]:
try:
    raw_files = [item.path for item in dbutils.fs.ls(RAW_PATH) if item.path.endswith('.json')]
except Exception:
    raw_files = []

if not raw_files:
    dbutils.notebook.exit(f'No hay archivos JSON en {RAW_PATH}')

raw_df = spark.read.schema(schema).option('multiLine', True).json(RAW_PATH)

bronze_df = (
    raw_df
    .withColumn('run_date', F.to_date('run_date'))
    .withColumn('valid_date', F.to_date('valid_date'))
    .withColumn('valid_datetime', F.to_timestamp('valid_datetime'))
    .withColumn('source_file', F.col('_metadata.file_path'))
    .withColumn('extracted_at_ts', F.to_timestamp('extracted_at'))
    .withColumn('ingestion_date', F.current_date())
    .withColumn('loaded_at', F.current_timestamp())
    .withColumn('updated_at', F.current_timestamp())
    .filter(F.col('run_date').isNotNull())
)

window = Window.partitionBy('run_date', 'run_time', 'step_hours', 'latitude', 'longitude').orderBy(F.col('extracted_at_ts').desc_nulls_last())
bronze_df = (
    bronze_df.withColumn('row_number', F.row_number().over(window))
    .filter(F.col('row_number') == 1)
    .drop('row_number')
    .select(
        'run_date', 'run_time', 'step_hours', 'valid_date', 'valid_datetime', 'latitude', 'longitude',
        'tp_mm', 'tipo', 'source_api', 'source_file', F.col('extracted_at_ts').alias('extracted_at'),
        'ingestion_date', 'loaded_at', 'updated_at',
    )
)

if bronze_df.limit(1).count() == 0:
    dbutils.notebook.exit('No valid rows to merge')

DeltaTable.forName(spark, BRONZE_TABLE).alias('t').merge(
    bronze_df.alias('s'),
    't.run_date = s.run_date AND t.run_time = s.run_time AND t.step_hours = s.step_hours AND t.latitude = s.latitude AND t.longitude = s.longitude',
).whenNotMatchedInsertAll().execute()

spark.table(BRONZE_TABLE).agg(F.min('run_date').alias('inicio'), F.max('run_date').alias('fin'), F.count('*').alias('rows')).show()
